In [6]:
from pathlib import Path
import pandas as pd
import yaml
import shutil

In [9]:
# Class mapping for YOLO
CLASS_NAMES = ["glass", "paper", "cardboard", "plastic", "metal"]
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}


def convert_bbox_to_yolo(width, height, xmin, ymin, xmax, ymax):
    """
    Convert Pascal VOC style box coordinates to YOLO format.

    YOLO format:
    class_id x_center y_center box_width box_height

    All values except class_id must be normalized to [0, 1].
    """
    x_center = ((xmin + xmax) / 2) / width
    y_center = ((ymin + ymax) / 2) / height
    box_width = (xmax - xmin) / width
    box_height = (ymax - ymin) / height
    return x_center, y_center, box_width, box_height


def prepare_split(split_dir: Path, output_root: Path):
    """
    Read _annotations.csv in a split directory (train/valid/test),
    create YOLO labels, and copy images into YOLO folder structure.
    """
    csv_path = split_dir / "_annotations.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {csv_path}")

    df = pd.read_csv(csv_path)

    split_name = split_dir.name
    images_out = output_root / "images" / split_name
    labels_out = output_root / "labels" / split_name

    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)

    # Group annotations by image
    grouped = df.groupby("filename")

    for filename, group in grouped:
        image_src = split_dir / filename
        image_dst = images_out / filename

        if not image_src.exists():
            print(f"Warning: image not found, skipping: {image_src}")
            continue

        # Copy image
        shutil.copy2(image_src, image_dst)

        # Write YOLO label file
        label_file = labels_out / f"{Path(filename).stem}.txt"

        lines = []
        for _, row in group.iterrows():
            class_name = str(row["class"]).strip().lower()

            if class_name not in CLASS_TO_ID:
                print(f"Warning: unknown class '{class_name}' in {filename}, skipping.")
                continue

            width = float(row["width"])
            height = float(row["height"])
            xmin = float(row["xmin"])
            ymin = float(row["ymin"])
            xmax = float(row["xmax"])
            ymax = float(row["ymax"])

            x_center, y_center, box_width, box_height = convert_bbox_to_yolo(
                width, height, xmin, ymin, xmax, ymax
            )

            class_id = CLASS_TO_ID[class_name]
            line = f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}"
            lines.append(line)

        with open(label_file, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))

    print(f"Finished preparing split: {split_name}")


def create_data_yaml(output_root: Path):
    """
    Create YOLO data.yaml file.
    """
    yaml_data = {
        "path": str(output_root.resolve()),
        "train": "images/train",
        "val": "images/valid",
        "test": "images/test",
        "names": CLASS_NAMES,
    }

    yaml_path = output_root / "data.yaml"
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(yaml_data, f, sort_keys=False, allow_unicode=True)

    print(f"Created YAML file: {yaml_path}")


def main():
    project_root = Path(__file__).resolve()
    data_root = project_root / "data"
    output_root = project_root / "yolo_dataset"

    for split in ["train", "valid", "test"]:
        split_dir = data_root / split
        if not split_dir.exists():
            raise FileNotFoundError(f"Split folder not found: {split_dir}")
        prepare_split(split_dir, output_root)

    create_data_yaml(output_root)

    print("\nAll done.")
    print(f"YOLO dataset created at: {output_root}")


if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined